In [1]:
pip install praw

  Using cached praw-7.7.0-py3-none-any.whl (189 kB)
  Using cached websocket_client-1.5.1-py3-none-any.whl (55 kB)
  Using cached prawcore-2.3.0-py3-none-any.whl (16 kB)
  Using cached update_checker-0.18.0-py3-none-any.whl (7.0 kB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
import praw
import json

creds = json.load(open("creds.json"))

username = creds.get("username")
clientid = creds.get("clientid")
clientsecret = creds.get("clientsecret")

reddit = praw.Reddit(client_id=clientid,
                     client_secret=clientsecret,
                     user_agent='data collection (by {})'.format(username),
                    redirect_uri='http://localhost:8000',)

In [4]:
import pandas as pd
from datetime import datetime

sreddits_names = ["dating_advice", "relationship_advice"]
counter = 0
posts_table = []
column_names = ["Index", "Subreddit", "Post time", 
                "User Name", "title", "text",
                "num_comments", "score", "ups", "upvote_ratio", 
                "downs", 'downvote', 
                "URL",  
               "over18"]

for subreddit in sreddits_names:
    results = reddit.subreddit(subreddit).new(limit=1000)
    for p in results: # p is a reddit post
        counter += 1
        # Add the post to the table
        posts_table.append([counter, p.subreddit, datetime.utcfromtimestamp(p.created), 
                            p.author, p.title, p.selftext, 
                            p.num_comments, p.score, p.ups, p.upvote_ratio, 
                            p.downs, p.downvote, 
                            p.permalink
                           , p.over_18])
    

df_posts = pd.DataFrame(posts_table, columns=column_names)

In [5]:
df_posts.to_csv("posts.csv", index=False) # index=False - do not save the default index column

In [7]:
# create a boolean mask for non-duplicated values
mask = ~df_posts['User Name'].duplicated(keep=False)

# filter the DataFrame using the boolean mask
df_filtered = df_posts.loc[mask]

# print the filtered DataFrame
print(len(df_filtered))

1736


In [8]:
df_filtered.to_csv("posts_wo_dups.csv", index=False) # index=False - do not save the default index column

In [ ]:
import pandas as pd

post_df = pd.read_csv("posts_with_score_bert.csv")
from googleapiclient import discovery
import json

API_KEY = 'xxxx'

client = discovery.build(
  "xxx",
  "xxx",
  developerKey=API_KEY,
  discoveryServiceUrl="xxxx",
  static_discovery=False,
)
rows = []

import time
from tqdm import tqdm

for post in tqdm(post_df['text'].values):
  analyze_request = {
    'comment': { 'text': post },
    'requestedAttributes': {'TOXICITY': {}, "THREAT": {}, "PROFANITY": {},
                            "INSULT":{}, "IDENTITY_ATTACK": {} }
  }
  try:
    response = client.comments().analyze(body=analyze_request).execute()
    TOXICITY_score = response.get("attributeScores").get("TOXICITY").get("summaryScore").get("value")
    THREAT_score = response.get("attributeScores").get("THREAT").get("summaryScore").get("value")
    PROFANITY_score = response.get("attributeScores").get("PROFANITY").get("summaryScore").get("value")
    INSULT_score = response.get("attributeScores").get("INSULT").get("summaryScore").get("value")
    IDENTITY_ATTACK_score = response.get("attributeScores").get("IDENTITY_ATTACK").get("summaryScore").get("value")
    time.sleep(3)
    row = [post, TOXICITY_score, THREAT_score, PROFANITY_score,INSULT_score, IDENTITY_ATTACK_score]
    rows.append(row)
  except Exception:
    continue

